## 3. Postprocessing Proxies Workflow

0. Packages
1. Comments
2. Settings
3. Download Data & Metadata
4. Merge Data & Metadata
5. Clipping & Masking
6. Reprojecting & Resampling
7. Conversion to NetCDF

### 0. Packages

In [13]:
# Packages
import affine
import geopandas as gpd
import pathlib
import glob
from google.cloud import storage
import matplotlib.pyplot as plt
import netCDF4 as nc4
import numpy as np
import os
import pandas as pd
from rasterio.enums import Resampling
import rioxarray as rxr
from shapely.geometry import Polygon
from tqdm import tqdm
import xarray as xr
from skimage.morphology import binary_erosion, binary_dilation, disk

### 1. Comments

Acknowledgements & code references:
- https://github.com/openearth/eo-bathymetry/
- https://github.com/openearth/eo-bathymetry-functions/
- https://github.com/gee-community/ee-packages-py

In [14]:
# TODO list
# TODO: add properties to the processed images again (CSV files from the cloud!)
# TODO: fix NC files to have correct headers & variable names
# TODO: merged product without interpolation & with interpolation (linear) over 500 m?
# TODO: standardize the NC header (interpolation flags & quality indicators)
# TODO: interpolated datasets; flag for interpolated points

### 2. Settings

In [24]:
# Settings
mode = 'intertidal_improved_100m_global'   # Specify mode, either 'intertidal' or 'subtidal'
start_date = '2021-01-01'                  # Start date of the composites
stop_date = '2022-01-01'                   # End date of the composites
compo_int = 12                             # Composite interval [months]
compo_len = 12                             # Composite length [months]
scale = 100                                # Output resolution of the image [m]
crs = 'EPSG:4326'                          # Output projection of the image

upscale = 100                              # Upscaling factor for the image
res_arc_min = 1/16                         # resolution of the image in arc minutes
res_deg = res_arc_min / 60                 # resolution of the image in degrees


# Tiling (see https://www.openearth.nl/rws-bathymetry/2019.html)
zoom_levels = [9, 10, 11] # list with zoom levels
zoom_level = zoom_levels[1] # zoom level to be used

# Directories and files
dir_path_base = r'p:\11209821-cmems-global-sdb'
dir_path_output = os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', f'{mode}')                                                         # Output directory
file_path_mask = os.path.join(dir_path_base, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result.parquet')                     # Mask file
file_path_mask_ed = os.path.join(dir_path_base, '00_miscellaneous', 'Feasibility_maps', '2024', 'gebco_2024_latminus2_merge_result_erosion_dilation.parquet') # Mask (erosion/dilation) file
file_path_tiles = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_polygons_world', f'df_boxes_world_Z{zoom_level}_filtered.parquet')                     # Tiles file
file_path_osm_water = os.path.join(dir_path_base, '01_intertidal', '02_data', '00_osm_files', 'geometries', 'osm_water_metadata.parquet')                     # OSM water metadata file
file_path_gebco = os.path.join(dir_path_base, '01_intertidal', '02_data', '03_gebco_files', 'gebco_2024.tif')                                                 # GEBCO merged file
file_path_credentials = os.path.join(dir_path_base, '00_miscellaneous', 'KEYS', 'cmems-sdb-11209821-002-d08744ac2a69.json')                                   # Cloud Storage credentials file
file_path_finaloutput = os.path.join(dir_path_base, '08_projectdeliverables')                                                                                 # Final output directory (after George merged all files)

# Google Cloud Bucket
bucket = 'cmems-isdb'

# Load Google credentials
if not file_path_credentials == '':  
    os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = file_path_credentials

### 3. Download Data & Metadata

In [16]:
# Create output directories
if not os.path.exists(os.path.join(dir_path_output, '01_data')):
    os.makedirs(os.path.join(dir_path_output, '01_data'))
if not os.path.exists(os.path.join(dir_path_output, '02_metadata')):
    os.makedirs(os.path.join(dir_path_output, '02_metadata'))

# Get all files from the bucket
client = storage.Client()
all_blobs = [blob for blob in client.list_blobs(bucket)]

# Get subset of files to download
for blob in tqdm(all_blobs, desc='Downloading files', unit='file'):
    # Get mode and zoom level
    mode_blob = blob.name.split('/')[0]
    zoom_level_blob = blob.name.split('/')[1]

    # Check if mode is correct
    if mode_blob == mode:
        file_path_data = os.path.join(dir_path_output, '01_data', '_'.join(blob.name.split('/')[1:]))
    elif mode_blob == '{}_meta'.format(mode):
        file_path_data = os.path.join(dir_path_output, '02_metadata', '_'.join(blob.name.split('/')[1:]))
    else:
        continue

        # Check if zoom level is correct
    if not zoom_level_blob == f'z{zoom_level}':
        continue

    # Check if file already exists
    if os.path.exists(file_path_data):
        #print(f'File already exists: {os.path.basename(file_path_data)}')
        continue

    # Download the file
    #print(f'Downloading: {os.path.basename(file_path_data)}')
    blob.download_to_filename(file_path_data)

### 4. Merge Data & Metadata

In [7]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '01_data', '*.tif'))
file_path_csvs = glob.glob(os.path.join(dir_path_output, '02_metadata', '*.csv'))

# Number of files
print('Number of tif files: {}'.format(len(file_path_tifs)))
print('Number of csv files: {}'.format(len(file_path_csvs)))

# Merge the files
for i, file_path_tif in enumerate(tqdm(file_path_tifs)):
    # Check if the merged file already exists
    file_path_merged_tif = os.path.join(dir_path_output, '03_merged', os.path.basename(file_path_tif))
    if os.path.exists(file_path_merged_tif):
        #print(f'Merged file already exists: {os.path.basename(file_path_merged_tif)}')
        continue 

    # Get the corresponding CSV file
    file_name_csv = os.path.join(dir_path_output, '02_metadata', os.path.basename(file_path_tif).replace('.tif', '.csv'))

    # Check if the CSV file exists
    if not os.path.exists(file_name_csv):
        #print('CSV does not exist')
        continue
    
    # Open tif and csv files
    da = rxr.open_rasterio(file_path_tif)
    csv_data = pd.read_csv(file_name_csv, index_col=0).iloc[0].to_dict()

    # Convert gtsm_station_temporal_offsets, gtsm_tidal_stage_percentages, gtsm_water_levels, quality_scores, system_time_starts to lists
    for key in ["gtsm_station_temporal_offsets", "gtsm_tidal_stage_percentages", "gtsm_water_levels", "quality_scores", "system_time_starts"]:
        csv_data[key] = eval(csv_data[key])

    # Convert gtsm_times to datetime
    import datetime
    csv_data['gtsm_times'] = csv_data['gtsm_times'].replace('[', '[\'').replace(', ', '\', \'').replace(']', '\']')
    csv_data['gtsm_times'] = eval(csv_data['gtsm_times'])
    csv_data['gtsm_times'] = [datetime.datetime.strptime(x[:-3], '%Y-%m-%dT%H:%M:%S.%f') for x in csv_data['gtsm_times']]

    # If data has been created using less than 10 images, skip it
    if len(csv_data['gtsm_times']) < 10:
        #print(f'Skipping {file_path_tif} due to insufficient data points.')
        da.close()
        continue
    
    # Convert system_time_starts to datetime
    csv_data['system_time_starts'] = [datetime.datetime.fromtimestamp(x/1000) for x in csv_data['system_time_starts']]

    # Convert gtsm_station_temporal_offsets to deltatime 
    from datetime import timedelta
    csv_data['gtsm_station_temporal_offsets'] = [timedelta(seconds=x/1000) for x in csv_data['gtsm_station_temporal_offsets']]

    # Add the csv data as attributes to tif
    for key, value in csv_data.items():
        da.attrs[key] = value
    
    # Scale second band between 1 and 5
    band = da.band.values[1]
    try:
        band_min = da.loc[dict(band=band)].min()
        band_max = da.loc[dict(band=band)].max()
    except Exception as e:
        print(f'Error reading band {band} from {file_path_tif}: {e}')
        da.close()
        continue
    da.loc[dict(band=band)] = ((da.loc[dict(band=band)] - band_min) / (band_max - band_min) * 4).round()
    
    # Plot the tif file
    plot = False
    if plot:
        fig, axs = plt.subplots(1, 2, figsize=(20, 10))
        da.isel(band=0).plot(ax=axs[0])
        da.isel(band=1).plot(ax=axs[1])

    # Create directory
    if not os.path.exists(os.path.join(dir_path_output, '03_merged')):
        os.makedirs(os.path.join(dir_path_output, '03_merged'))

    # Save the tif file
    file_path_merged_tif = os.path.join(dir_path_output, '03_merged', os.path.basename(file_path_tif))
    da.rio.to_raster(file_path_merged_tif, driver='GTiff', compress="LZW")
    da.close()

Number of tif files: 12474
Number of csv files: 13236


 85%|████████▌ | 10661/12474 [00:24<00:06, 276.78it/s]

Error reading band 2 from p:\11209821-cmems-global-sdb\01_intertidal\02_data\05_calibrated\intertidal_improved_100m_global\01_data\z10_x1004_y638_t2021-01-01_2022-01-01_100m.tif: Read or write failed. p:/11209821-cmems-global-sdb/01_intertidal/02_data/05_calibrated/intertidal_improved_100m_global/01_data/z10_x1004_y638_t2021-01-01_2022-01-01_100m.tif, band 2: IReadBlock failed at X offset 0, Y offset 0: TIFFReadEncodedTile() failed.


100%|██████████| 12474/12474 [00:29<00:00, 427.85it/s]


<h1 style="color:red;"> TEMPORARY CODE</h1>

In [ ]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '01_data', '*.tif'))

# Number of files
print('Number of tif files: {}'.format(len(file_path_tifs)))

# Merge the files
for i, file_path_tif in enumerate(tqdm(file_path_tifs)):
    # Open tif and csv files
    da = rxr.open_rasterio(file_path_tif)
    
    # Scale second band between 1 and 5
    band = da.band.values[1]
    band_min = da.loc[dict(band=band)].min()
    band_max = da.loc[dict(band=band)].max()
    da.loc[dict(band=band)] = ((da.loc[dict(band=band)] - band_min) / (band_max - band_min) * 4).round()
    
    # Plot the tif file
    plot = False
    if plot:
        fig, axs = plt.subplots(1, 2, figsize=(20, 10))
        da.isel(band=0).plot(ax=axs[0])
        da.isel(band=1).plot(ax=axs[1])

    # Create directory
    if not os.path.exists(os.path.join(dir_path_output, '03_merged')):
        os.makedirs(os.path.join(dir_path_output, '03_merged'))

    # Save the tif file9
    file_path_merged_tif = os.path.join(dir_path_output, '03_merged', os.path.basename(file_path_tif))
    da.rio.to_raster(file_path_merged_tif, driver='GTiff', compress="LZW")
    da.close()

### 5. Clipping & Masking

In [8]:
# Read geometries
gdf_mask = gpd.read_parquet(file_path_mask)
gdf_mask_ed = gpd.read_parquet(file_path_mask_ed)
gdf_tiles = gpd.read_parquet(file_path_tiles)
gdf_osm_water = gpd.read_parquet(file_path_osm_water)
da_gebco = rxr.open_rasterio(file_path_gebco)

In [9]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '03_merged', '*.tif'))
# Sort files
file_path_tifs = sorted(file_path_tifs)

# Filter tiles based on name
names = ['_'.join(os.path.basename(x).split('_')[:3]) for x in file_path_tifs]
gdf_tiles_ss = gdf_tiles[gdf_tiles['name'].isin(names)]

# Filter files based on name
file_path_tifs = [x for x in file_path_tifs if '_'.join(os.path.basename(x).split('_')[:3]) in gdf_tiles_ss['name'].values]

# Sort tiles according to names
gdf_tiles_ss = gdf_tiles_ss.sort_values(by='name')

# Add files to tiles
gdf_tiles_ss['file_path_tif'] = file_path_tifs

# Number of files and tiles
print('Number of tif files and tiles: {}'.format(len(file_path_tifs)))

Number of tif files and tiles: 12381


In [13]:
for i, (idx, row) in enumerate(tqdm(gdf_tiles_ss.iterrows(), total=len(gdf_tiles_ss))):
    # Select a specific tile
    # Waddenzee: z10_x527_y332
    # Le Mont-Saint-Michel: z10_x507_y353
    # Estuary: z10_x509_y366
    # Rochfort: z10_x508_y364
    # La Rochelle: z10_x506_y358
    # Shark Bay: z10_x834_y583
    #if row['name'] != 'z10_x834_y583':
    #    continue

    # Check directory
    if not os.path.exists(os.path.join(dir_path_output, '04_clipped')):
        os.makedirs(os.path.join(dir_path_output, '04_clipped'))

    # Check if the merged file already exists
    file_path_clipped_tif = os.path.join(dir_path_output, '04_clipped', os.path.basename(row['file_path_tif']))
    if os.path.exists(file_path_clipped_tif):
        #print(f'Merged file already exists: {os.path.basename(file_path_merged_tif)}')
        continue 

    # Open the tif file
    da = rxr.open_rasterio(row['file_path_tif'])

    # Clip the tif file to the tile
    #da_clipped = da.rio.clip([row['geometry']], da.rio.crs, drop=True)
    da_clipped = da.copy()

    # Clip gebco data to tile
    bounds = da.rio.bounds()
    da_gebco_clipped = da_gebco.rio.clip_box(bounds[0], bounds[1], bounds[2], bounds[3], crs=da_gebco.rio.crs)

    # Reproject match gebco data to tile
    da_gebco_clipped = da_gebco_clipped.rio.reproject_match(da_clipped, resampling=Resampling.bilinear)

    # Set nodata value to nan
    da_gebco_clipped = da_gebco_clipped.where(da_gebco_clipped != da_gebco_clipped.rio.nodata)
    da_gebco_clipped = da_gebco_clipped.rio.write_nodata(np.nan)

    # Get the gebco mask (gebco between min and max of results)
    threshold = 1
    da_gebco_mask = (((da_gebco_clipped.isel(band=0) + threshold) > da_clipped.isel(band=0).min()) &
                     ((da_gebco_clipped.isel(band=0) - threshold) < da_clipped.isel(band=0).max()))

    # Get osm water files that intersect with the tile
    gdf_osm_water_tile = gdf_osm_water[gdf_osm_water.intersects(row['geometry'])]

    # Read osm water files
    da_osm_water_ls = [rxr.open_rasterio(file_path).isel(band=0) for file_path in gdf_osm_water_tile['file_path']]
    
    # Reproject match the osm water files to the tif file
    da_osm_water_ls_reprojected = [da_osm_water.rio.reproject_match(da_clipped) for da_osm_water in da_osm_water_ls]

    # Set nodata value to nan
    da_osm_water_ls_reprojected = [da.where(da != da.rio.nodata) for da in da_osm_water_ls_reprojected]
    da_osm_water_ls_reprojected = [da.rio.write_nodata(np.nan) for da in da_osm_water_ls_reprojected]
    if da_osm_water_ls_reprojected == []:
        print('{}: No OSM water file to mask with'.format(row['name']))
        continue

    # Merge the osm water files
    da_osm_water = xr.concat(da_osm_water_ls_reprojected, dim='band').max(dim='band')

    # Create a mask
    da_osm_mask = (da_osm_water == 1) | (da_osm_water == 2)

    # Morpohological erosion and dilation
    
    da_osm_mask_ed = da_osm_mask.copy()
    da_osm_mask_ed.values = binary_dilation(binary_erosion(da_osm_mask.values, disk(5)), disk(5))

    # Mask the tif file
    da_masked_gebcosm = da_clipped.where(da_gebco_mask & da_osm_mask_ed)

    # Percentile mask
    da_mask_perc = da_masked_gebcosm.isel(band=0).copy()
    percentiles = [1, 99]
    if not np.isnan(da_mask_perc.values).all():
        cutoffs = np.nanpercentile(da_mask_perc.values, percentiles)
    else:
        print('{}: Contains only nan values after masking'.format(row['name']))
        continue

    da_mask_perc.values = (da_mask_perc > cutoffs[0]) & (da_mask_perc < cutoffs[1])

    # Morphological erosion and dilation
    da_mask_perc_ed = da_mask_perc.copy()
    da_mask_perc_ed.values = binary_dilation(binary_erosion(da_mask_perc.values, disk(2)), disk(2))
    
    # Clip the tif file
    da_masked_perc = da_clipped.where(da_mask_perc_ed)
    if da_masked_perc.isnull().all():
        print('{}: Contains only nan values after masking'.format(row['name']))
        da.close()
        continue
        
    # Save the tif file
    da_masked_perc.rio.to_raster(file_path_clipped_tif, driver="GTiff", compress="LZW")
    
    # Close the datasets
    da.close()
    da_clipped.close()
    da_gebco_clipped.close()
    da_osm_water.close()
    da_osm_mask.close()
    da_osm_mask_ed.close()
    da_masked_gebcosm.close()
    da_mask_perc.close()
    da_mask_perc_ed.close()
    da_masked_perc.close()

  0%|          | 2/12381 [00:01<1:50:57,  1.86it/s]

z10_x1000_y651: Contains only nan values after masking


  0%|          | 10/12381 [00:01<24:55,  8.27it/s] 

z10_x1003_y334: Contains only nan values after masking


  0%|          | 18/12381 [00:01<17:26, 11.82it/s]

...

100%|█████████▉| 12375/12381 [1:50:25<00:03,  1.64it/s]

z10_x999_y491: Contains only nan values after masking


100%|█████████▉| 12378/12381 [1:50:27<00:01,  1.72it/s]

z10_x999_y652: Contains only nan values after masking


100%|██████████| 12381/12381 [1:50:30<00:00,  1.87it/s]


In [ ]:
fig, axs = plt.subplots(4, 3, figsize=(16, 10))
axs = axs.flatten()
da_clipped.isel(band=0).plot.imshow(ax=axs[0], center=False, vmin=da_clipped.min(), vmax=da_clipped.max())
da_gebco_clipped.isel(band=0).plot.imshow(ax=axs[1], center=False)
da_gebco_mask.plot.imshow(ax=axs[2])
da_osm_water.plot.imshow(ax=axs[3], center=False)
da_osm_mask.plot.imshow(ax=axs[4])
da_osm_mask_ed.plot.imshow(ax=axs[5])
da_masked_gebcosm.isel(band=0).plot.imshow(ax=axs[6], center=False)
da_mask_perc.plot.imshow(ax=axs[7], center=False)
da_mask_perc_ed.plot.imshow(ax=axs[8])
da_masked_perc.isel(band=0).plot.imshow(ax=axs[9], center=False)
for ax in axs:
    ax.set_aspect('equal')
fig.tight_layout()

In [ ]:
import contextily as ctx
fig, ax = plt.subplots(figsize=(10, 6))
da_masked_perc.isel(band=0).plot.imshow(ax=ax, center=False, cmap='Spectral_r')
ctx.add_basemap(ax, crs=da_masked_perc.rio.crs, zorder=-1, source=ctx.providers.Esri.WorldImagery, attribution=False)

### 6. Reprojecting & Resampling

In [14]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '04_clipped', '*.tif'))

# Number of files
print('Number of tif files: {}'.format(len(file_path_tifs)))

# Reproject the files
for i, file_path_tif in enumerate(tqdm(file_path_tifs)):
    # Select specific file
    #if i != 100:
    #    continue

    # Open tif file
    da = rxr.open_rasterio(file_path_tif)

    # Get bounds
    gdf_bounds = gpd.GeoDataFrame(geometry=[Polygon.from_bounds(*da.rio.bounds())], crs=da.rio.crs)

    # Reproject bounds
    gdf_bounds_reprojected = gdf_bounds.to_crs(crs)

    # Reproject the tif file
    reproject_type = 'transform'
    if reproject_type == 'scale_factor':
        scale_factor = upscale / scale
        da_reprojected = da.rio.reproject(crs, shape=(int(da.rio.height / scale_factor), int(da.rio.width / scale_factor)))
    elif reproject_type == 'resolution':
        da_reprojected = da.rio.reproject(crs, resolution=res_deg, nodata=np.nan)
    elif reproject_type == 'transform':
        shift = (0, 0) # (0.5, 0.5)
        tlx = (round(gdf_bounds_reprojected.total_bounds[0]/res_deg - shift[0]) + shift[0]) * res_deg
        tly = (round(gdf_bounds_reprojected.total_bounds[3]/res_deg - shift[1]) + shift[1]) * res_deg
        transform = affine.Affine(res_deg, 0, tlx, 0, -res_deg, tly)
        da_reprojected1 = da.isel(band=0).rio.reproject(crs, transform=transform, nodata=np.nan, resampling=Resampling.bilinear)
        da_reprojected2 = da.isel(band=1).rio.reproject(crs, transform=transform, nodata=np.nan, resampling=Resampling.nearest)
        da_reprojected = xr.concat([da_reprojected1, da_reprojected2], dim='band')
    
    # Mask the reprojected data
    da_reprojected = da_reprojected.where(da_reprojected.isel(band=0).notnull())

    # Check if the reprojected data is empty
    if da_reprojected.isel(band=0).isnull().all():
        print('{}: Contains only nan values after reprojection'.format(os.path.basename(file_path_tif)))
        da.close()
        continue
    
    # Plot the tif file
    plot = False
    if plot:
        fig, axs = plt.subplots(2, 2, figsize=(20, 20))
        axs = axs.flatten()
        da.isel(band=0).plot(ax=axs[0])
        da.isel(band=1).plot(ax=axs[1])
        da_reprojected.isel(band=0).plot(ax=axs[2])
        da_reprojected.isel(band=1).plot(ax=axs[3])
        fig.tight_layout()
    
    # Check directory
    if not os.path.exists(os.path.join(dir_path_output, '05_reprojected')):
        os.makedirs(os.path.join(dir_path_output, '05_reprojected'))
        
    # Save the tif file
    file_path_reprojected_tif = os.path.join(dir_path_output, '05_reprojected', "_".join(os.path.basename(file_path_tif).split("_")[:-1]) + ".tif")
    da_reprojected.rio.to_raster(file_path_reprojected_tif, driver="GTiff", compress="LZW")
    da.close()
    da_reprojected.close()

Number of tif files: 9534


100%|██████████| 9534/9534 [33:43<00:00,  4.71it/s]


### 7. Conversion to NetCDF

In [15]:
# Get files
file_path_tifs = glob.glob(os.path.join(dir_path_output, '05_reprojected', '*.tif'))

# Number of files
print('Number of tif files: {}'.format(len(file_path_tifs)))

# Loop over the files
for i, file_path_tif in tqdm(enumerate(file_path_tifs), total=len(file_path_tifs)):    
    # Open tif file
    da = rxr.open_rasterio(file_path_tif)

    # Remove spatial ref
    crs_out = da.rio.crs
    da = da.drop_vars('spatial_ref')

    # Convert data array to dataset
    da = da.assign_coords(band=['elevation', 'Quality_indicator'])
    ds = da.to_dataset(dim='band')

    # Rename x and y to lon and lat
    ds = ds.rename({'x': 'lon', 'y': 'lat'}) 

    # Add variables
    ds['SDB_type'] = xr.DataArray(np.zeros_like(ds['elevation']), dims=('lat', 'lon'), coords={'lat': ds['lat'], 'lon': ds['lon']})
    ds['crs'] = int(crs_out.to_epsg())

    # Replace nan values with 127
    ds['Quality_indicator'] = ds['Quality_indicator'].where(~np.isnan(ds['elevation']),127)
    ds['SDB_type'] = ds['SDB_type'].where(~np.isnan(ds['elevation']), 127)

    # Convert types
    ds['lon'] = ds['lon'].astype('float64')
    ds['lat'] = ds['lat'].astype('float64')
    ds['elevation'] = ds['elevation'].astype('float32')
    ds['Quality_indicator'] = ds['Quality_indicator'].astype('byte')
    ds['SDB_type'] = ds['SDB_type'].astype('byte')
    ds['crs'] = ds['crs'].astype('int64')

    # Remove attributes
    ds.lon.attrs = {'axis': 'X', 'long_name': 'longitude', 'standard_name': 'longitude', 'units': 'degrees_east'}
    ds.lat.attrs = {'axis': 'Y', 'long_name': 'latitude', 'standard_name': 'latitude', 'units': 'degrees_north'}
    ds.elevation.attrs = {'long_name': 'Satellite derived bathymetric depth value',
                          'standard_name': 'depth',
                          'units': 'm',
                          'grid_mapping': 'crs'}
    ds.Quality_indicator.attrs = {'_FillValue': np.byte(127),
                                  'long_name': 'Unified qualitative representation of quality',
                                  'standard_name': 'quality_flag',
                                  'valid_range': [np.byte(0), np.byte(4)],
                                  'flag_values': np.array([0, 1, 2, 3, 4], dtype='byte'),
                                  'flag_meaning': 'Highly_reliable Reliable Moderately_reliable Unreliable Highly_unreliable',
                                  'grid_mapping': 'crs'}
    ds.SDB_type.attrs = {'_FillValue': np.byte(127),
                         'long_name': 'SDB methodology used for elevation value',
                         'standard_name': 'SDB_methodology',
                         'valid_range': [np.byte(0), np.byte(4)],
                         'flag_values': np.array([0, 1, 2, 3, 4], dtype='byte'),
                         'flag_meaning': 'Intertidal_Bathymetry Multi_spectral_SDB Wave_kinematics Interpolated ground_truth_control_point',
                         'grid_mapping': 'crs'}
    ds.crs.attrs = {'grid_mapping_name': 'latitude_longitude', 'long_name': 'grid mapping', 'epsg_code': str(crs)}
    ds.attrs = {'AREA_OR_POINT': 'Area',
                'STATISTICS_APPROXIMATE': 'YES',
                'STATISTICS_MAXIMUM': ds['elevation'].max().values,
                'STATISTICS_MEAN': ds['elevation'].mean().values,
                'STATISTICS_MINIMUM': ds['elevation'].min().values,
                'STATISTICS_STDDEV': ds['elevation'].std().values,
                'STATISTICS_VALID_PERCENT': (ds['elevation'].count() / ds['elevation'].size * 100).values,
                'scale_factor': 1.0,
                'add_offset': 0.0,
                '_FillValue': np.nan,
                'dtm_convention_version': '1.0',
                'Conventions': 'SeaDataNet_1.0 CF1.6',
                'title': 'Provision of global coastal bathymetry derived from Sentinel 2 observations',
                'institution': 'On behalf of the Copernicus Marine Project, https://marine.copernicus.eu/',
                'source': 'Sentinel 2 observations',
                'comment': 'This data should not be used for navigation or any purpose relating to safety at sea.',
                'history': 'Created with Python script in June 2025'}
    
    # Check directory
    if not os.path.exists(os.path.join(dir_path_output, '06_netcdf')):
        os.makedirs(os.path.join(dir_path_output, '06_netcdf'))
    
    # Write to netcdf
    file_path_netcdf = os.path.join(dir_path_output, '06_netcdf', os.path.basename(file_path_tif).replace('.tif', '.nc'))
    ds.to_netcdf(file_path_netcdf, mode="w", format='NetCDF4')

    # Read netcdf
    cdf = False
    if cdf:
        with nc4.Dataset(file_path_netcdf) as nc:
            # Write header file (cdl)
            file_path_cdl = file_path_netcdf.replace(".nc", ".cdl")
            cdl = nc.tocdl(file_path_cdl)
            with open(file_path_cdl, 'w') as f:
                f.write(cdl)

Number of tif files: 9534


100%|██████████| 9534/9534 [29:44<00:00,  5.34it/s]


### 8. Upload to AWS S3 bucket

In [3]:
# Packages
import dotenv
import boto3

# File paths
#file_path_env = r'C:\Users\white_rn\Documents\GitHub\eo-bathymetry\.env'
file_path_env = os.path.join(pathlib.Path.cwd().parent.parent, ".env")
dir_path_nc = os.path.join(dir_path_output, '06_netcdf')
bucket_name = 'global-sdb-data-delivery'
prefix = 'intertidal_bathymetry_v0/'

# Load environment variables
dotenv.load_dotenv(file_path_env)

# AWS S3 client
s3 = boto3.client(
    's3',
    aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'))

# List objects in the S3 bucket
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

In [93]:
# Function to upload a folder to S3
def upload_folder(dir_path, bucket, prefix=''):
    for dir_root, _, files in os.walk(dir_path):
        for file in tqdm(files, desc=f'Uploading files', unit='file'):
            file_path = os.path.join(dir_root, file)
            relative_path = os.path.relpath(file_path, dir_path)
            s3_key = os.path.join(prefix, relative_path)
            s3.upload_file(file_path, bucket, s3_key)

# Upload the folder to S3
upload_folder(dir_path_nc, bucket_name, prefix)

Uploading files: 100%|██████████| 9534/9534 [23:40<00:00,  6.71file/s]


In [7]:
# List prefixed objects in the S3 bucket
response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

print("Folders ({}):".format(len(response.get('CommonPrefixes', []))))
for i, folder in enumerate(response.get('CommonPrefixes', [])):
    print('- {}'.format(folder['Prefix']))
    if i >= 3:
        print('- ...')
        break

print("Files ({}):".format(len(response.get('Contents', []))))
for i, file in enumerate(response.get('Contents', [])):
    print('- {}'.format(file['Key']))
    if i >= 3:
        print('- ...')
        break

Folders (0):
Files (1000):
- intertidal_bathymetry_v0/z10_x1000_y642_t2021-01-01_2022-01-01.nc
- intertidal_bathymetry_v0/z10_x1000_y652_t2021-01-01_2022-01-01.nc
- intertidal_bathymetry_v0/z10_x1001_y640_t2021-01-01_2022-01-01.nc
- intertidal_bathymetry_v0/z10_x1001_y641_t2021-01-01_2022-01-01.nc
- ...


In [8]:
# List all objects in the bucket
response = s3.list_objects_v2(Bucket=bucket_name)

print("Folders ({}):".format(len(response.get('CommonPrefixes', []))))
for i, folder in enumerate(response.get('CommonPrefixes', [])):
    print('- {}'.format(folder['Prefix']))
    if i >= 3:
        print('- ...')
        break

print("Files ({}):".format(len(response.get('Contents', []))))
for i, file in enumerate(response.get('Contents', [])):
    print('- {}'.format(file['Key']))
    #if i >= 3:
        #print('- ...')
        #break

Folders (0):
Files (1000):
- Final/
- Final/Intertidal/
- Final/Intertidal/Global SDB intertidal v1.nc
- Final/RTE/
- Final/RTE/Global SDB RTE v2.nc
- Final/Wave /
- Final/Wave /Global SDB Wave v3.nc
- Project data deliverables/
- Project data deliverables/Global SDB 2025 RTE.nc
- Project data deliverables/Global SDB 2025 Wave.nc
- Project data deliverables/Global SDB 2025 combined.nc
- Project data deliverables/Global SDB 2025 intertidal.nc
- RTE/global/RTE_01GEL.nc
- RTE/global/RTE_01GEM.nc
- RTE/global/RTE_01JDH.nc
- RTE/global/RTE_01KBB.nc
- RTE/global/RTE_01KCA.nc
- RTE/global/RTE_01KCB.nc
- RTE/global/RTE_01KCS.nc
- RTE/global/RTE_01KCT.nc
- RTE/global/RTE_01KCU.nc
- RTE/global/RTE_01KCV.nc
- RTE/global/RTE_01KFS.nc
- RTE/global/RTE_01KFT.nc
- RTE/global/RTE_01KGT.nc
- RTE/global/RTE_01KGV.nc
- RTE/global/RTE_01KHV.nc
- RTE/global/RTE_01LAC.nc
- RTE/global/RTE_01LBC.nc
- RTE/global/RTE_01LCE.nc
- RTE/global/RTE_01LEF.nc
- RTE/global/RTE_01LFC.nc
- RTE/global/RTE_01LHC.nc
- RTE/gl

In [23]:
from collections import Counter

def list_folder_prefixes(bucket_name, prefix='', depth=1):
    paginator = s3.get_paginator('list_objects_v2')

    folder_counter = Counter()

    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if 'Contents' in page:
            for obj in page['Contents']:
                key = obj['Key']
                parts = key.split('/')
                if len(parts) >= depth:
                    folder = '/'.join(parts[:depth]) + '/'
                    folder_counter[folder] += 1

    return folder_counter

# Example usage
depth = 1  # How deep you want to consider as "folder", e.g., 'data/2023/'
folder_counts = list_folder_prefixes(bucket_name, prefix="", depth=depth)

for folder, count in folder_counts.items():
    print(f"{folder}: {count} objects")

Final/: 7 objects
Project data deliverables/: 5 objects
RTE/: 5196 objects
Testdata 30 May 2025/: 2 objects
WKB_new/: 8386 objects
intertidal_bathymetry/: 9 objects
intertidal_bathymetry\z10_x1000_y642_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry\z10_x1000_y652_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry\z10_x1001_y640_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry\z10_x1001_y641_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry\z10_x1001_y651_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry\z10_x1002_y638_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry\z10_x1002_y650_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry\z10_x1002_y651_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry\z10_x1003_y507_t2021-01-01_2022-01-01.nc/: 1 objects
intertidal_bathymetry_20250611/: 9534 objects
intertidal_bathymetry_20250612/: 388 objects
intertidal_bathymetry_20250623/: 1971 objects
intertidal_bathymetry_v

In [58]:
# download files from the bucket
download_prefix = 'Project data deliverables/'

# List and download all objects under the prefix
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=bucket_name, Prefix=download_prefix):
    if 'Contents' in page:
        for obj in page['Contents']:
            key = obj['Key']

            # Skip 'folder' objects (common in S3 when folders are empty)
            if key.endswith('/'):
                continue

            print(f"Processing {key}")

            # Get the relative path after the prefix, comment out to preserve download_prefix folder struct
            relative_path = os.path.relpath(key, download_prefix) if download_prefix else key
            print(relative_path)

            # Build the full local path using OS separators
            local_file_path = os.path.join(file_path_finaloutput, *key.split('/'))
            print(key, local_file_path)

            # Create any necessary subdirectories
            os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
            print(os.path.dirname(local_file_path))

            # Download the file
            s3.download_file(bucket_name, key, local_file_path)
            print(f"Downloaded {key} to {local_file_path}")

Processing Project data deliverables/Global SDB 2025 RTE.nc
Project data deliverables/Global SDB 2025 RTE.nc p:\11209821-cmems-global-sdb\08_projectdeliverables\Project data deliverables\Global SDB 2025 RTE.nc
p:\11209821-cmems-global-sdb\08_projectdeliverables\Project data deliverables
Downloaded Project data deliverables/Global SDB 2025 RTE.nc to p:\11209821-cmems-global-sdb\08_projectdeliverables\Project data deliverables\Global SDB 2025 RTE.nc
Processing Project data deliverables/Global SDB 2025 Wave.nc
Project data deliverables/Global SDB 2025 Wave.nc p:\11209821-cmems-global-sdb\08_projectdeliverables\Project data deliverables\Global SDB 2025 Wave.nc
p:\11209821-cmems-global-sdb\08_projectdeliverables\Project data deliverables
Downloaded Project data deliverables/Global SDB 2025 Wave.nc to p:\11209821-cmems-global-sdb\08_projectdeliverables\Project data deliverables\Global SDB 2025 Wave.nc
Processing Project data deliverables/Global SDB 2025 combined.nc
Project data deliverables/

In [91]:
# Delete objects in the bucket - DOESNT WORK!
# response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

# if 'Contents' in response:
#     # Prepare keys to delete
#     objects_to_delete = [{'Key': obj['Key']} for obj in response['Contents']]
#     print(objects_to_delete)

#     # Delete them
#     s3.delete_objects(Bucket=bucket_name, Delete={'Objects': objects_to_delete})
#     print(f"Deleted all files in folder: {prefix}")
# else:
#     print(f"No files found in folder: {prefix}")